# Kaggle — dataset redimensionado `melanoma-isic2020-512` (F2.0)

Corre **una sola vez**. Antes de correr:

1. **Add Data → Competitions → `SIIM-ISIC Melanoma Classification`** (imágenes originales, `jpeg/train/`).
2. **Add Data → Your Datasets → `melanoma-isic2020-splits`** (manifiesto con `sha256_resized`).
3. Sin GPU (es trabajo de CPU) y con Internet activado (clona el repositorio).

Qué hace: clona el repositorio en un commit fijo, lee `resize.long_side` y `resize.jpeg_quality` de
`configs/data/isic2020.yaml` (no se escriben a mano), redimensiona las 33,126 imágenes con
`melanoma.data.resize.resize_many` (el mismo código de F1), calcula el SHA256 de cada resultado y lo
compara con `sha256_resized` del manifiesto local. **Reporta cuántas coinciden.** Si no coinciden
todas no es un fallo: se documenta la diferencia y su causa probable (versión de Pillow/libjpeg).

Al final, publica `/kaggle/working/isic2020_512` como dataset privado con la API de Kaggle
(requiere el secreto `KAGGLE_KEY` en *Add-ons → Secrets*), o bien: **Save Version → Output → New Dataset**.


In [ ]:
# Parámetros. REPO_SHA: commit del repositorio que se ejecuta (no `main`).
REPO_URL = "https://github.com/Edgar-Ontiveros/melanoma-triage.git"
REPO_SHA = "main"  # ← sustituir por el SHA de la corrida
COMPETITION_DIR = "/kaggle/input/competitions/siim-isic-melanoma-classification/jpeg/train"
SPLITS_DIR = "/kaggle/input/datasets/edgaronti26/melanoma-isic2020-splits"
OUT_DIR = "/kaggle/working/isic2020_512"
DATASET_ID = "edgaronti26/melanoma-isic2020-512"
WORKERS = 4

In [ ]:
import os
import subprocess
import sys

subprocess.run(["git", "clone", "--quiet", REPO_URL, "/kaggle/working/melanoma"], check=True)
subprocess.run(
    ["git", "-C", "/kaggle/working/melanoma", "checkout", "--quiet", REPO_SHA], check=True
)
os.chdir("/kaggle/working/melanoma")
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
# Solo el grupo core: el redimensionado no necesita torch.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import PIL
from omegaconf import OmegaConf

from melanoma.data.resize import resize_many

cfg = OmegaConf.load("configs/data/isic2020.yaml")
LONG_SIDE, QUALITY = int(cfg.resize.long_side), int(cfg.resize.jpeg_quality)
print(f"Pillow {PIL.__version__} · long_side={LONG_SIDE} · jpeg_quality={QUALITY}")

manifest = pd.read_csv(f"{SPLITS_DIR}/isic2020.csv", dtype={"image_id": str, "patient_id": str})
assert "sha256_resized" in manifest.columns, (
    "el manifiesto del dataset de splits no tiene sha256_resized"
)
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)
pairs = [(Path(COMPETITION_DIR) / f"{i}.jpg", out / f"{i}.jpg") for i in manifest["image_id"]]
missing = [s for s, _ in pairs if not s.exists()]
assert not missing, f"faltan {len(missing)} originales en la competencia: {missing[:5]}"

t0 = time.time()
results = resize_many(pairs, long_side=LONG_SIDE, quality=QUALITY, workers=WORKERS)
print(f"{len(results)} imágenes redimensionadas en {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Verificación contra las huellas locales (F2.0, paso 4-5)
manifest["sha256_kaggle"] = [r["sha256"] for r in results]
manifest["width_kaggle"] = [r["width"] for r in results]
manifest["height_kaggle"] = [r["height"] for r in results]
match = manifest["sha256_kaggle"] == manifest["sha256_resized"]
same_dims = (manifest["width_kaggle"] == manifest["width_resized"]) & (
    manifest["height_kaggle"] == manifest["height_resized"]
)
n = len(manifest)
report = {
    "pillow_kaggle": PIL.__version__,
    "long_side": LONG_SIDE,
    "jpeg_quality": QUALITY,
    "images": n,
    "sha256_match": int(match.sum()),
    "sha256_match_pct": round(100 * match.mean(), 2),
    "dims_match": int(same_dims.sum()),
}
print(json.dumps(report, indent=2))
Path("/kaggle/working/resize_verification.json").write_text(json.dumps(report, indent=2))
manifest.loc[
    ~match, ["image_id", "sha256_resized", "sha256_kaggle", "width_resized", "width_kaggle"]
].head(20)

## Interpretación

- **Todas coinciden:** la copia de Kaggle es byte-idéntica a `data/processed/isic2020_512` de la laptop.
- **No todas coinciden pero las dimensiones sí:** el redimensionado es el mismo (mismo código, mismos
  parámetros); difiere la codificación JPEG por versión de Pillow/libjpeg. La procedencia sigue siendo
  válida porque F1 verificó los **originales** byte a byte (200/200). Anotar el porcentaje, la versión
  de Pillow de Kaggle y la local (`uv run python -c "import PIL; print(PIL.__version__)"`) en
  `docs/DATA.md` y en `docs/specs/F2.md` (desviaciones).
- **Dimensiones distintas:** algo cambió en el código o en los parámetros. Detener y revisar el SHA del repositorio.


In [ ]:
# Publicar como dataset privado con la API de Kaggle. Necesita el secreto KAGGLE_KEY
# (Add-ons → Secrets). Alternativa sin API: Save Version → Output → "New Dataset"
# desde /kaggle/working/isic2020_512.
from kaggle_secrets import UserSecretsClient

os.environ["KAGGLE_USERNAME"] = DATASET_ID.split("/")[0]
os.environ["KAGGLE_KEY"] = UserSecretsClient().get_secret("KAGGLE_KEY")
meta = {
    "title": "melanoma-isic2020-512",
    "id": DATASET_ID,
    "licenses": [{"name": "CC-BY-NC-SA-4.0"}],
    "subtitle": "ISIC 2020 redimensionado a 512 px (JPEG q95) para el proyecto melanoma",
    "description": (
        "Imagenes del SIIM-ISIC 2020 Challenge Dataset (CC-BY-NC 4.0, "
        f"https://doi.org/10.34970/2020-ds01) redimensionadas al lado largo de {LONG_SIDE} px "
        f"con JPEG calidad {QUALITY}, con el codigo de {REPO_URL} (commit {REPO_SHA}). "
        f"Verificacion de huellas: {report['sha256_match']}/{n} coinciden con la copia local."
    ),
    "keywords": ["medicine", "image"],
}
Path(OUT_DIR, "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
Path(OUT_DIR, "resize_verification.json").write_text(json.dumps(report, indent=2))
subprocess.run(["kaggle", "datasets", "create", "-p", OUT_DIR, "--dir-mode", "skip"], check=True)
print(f"https://www.kaggle.com/datasets/{DATASET_ID}")